# BÀI 1: HIỂU DỮ LIỆU, CHỌN DỮ LIỆU VÀ TIỀN XỬ LÝ

## Tập dữ liệu D3 - Bank Marketing

### 1. Đề xuất bộ dữ liệu và ma trận kỹ thuật

#### 1.1. Nguồn gốc, giấy phép, quy mô

- **Tên dataset:** Bank Marketing
- **Nguồn:** UCI Machine Learning Repository
- **Link:** https://archive.ics.uci.edu/dataset/222/bank+marketing
- **Tác giả:** S. Moro, P. Rita, P. Cortez
- **Giấy phép:** CC BY 4.0
- **Nội dung:** Dữ liệu các chiến dịch marketing trực tiếp của ngân hàng.
- **Quy mô công bố:** 45.211 bản ghi, 30 thuộc tính đầu vào và 1 thuộc tính mục tiêu (`y`).
- **Tổng số cột ban đầu:** 17 cột, gồm 16 thuộc tính đầu vào và biến mục tiêu `y`.

#### 1.2. Tri thức lĩnh vực và kỹ thuật

Mục tiêu `y` cho biết khách hàng có đăng ký tiền gửi có kỳ hạn hay không.

- **Phân lớp:** Dự đoán `y = yes/no`.
- **Luật kết hợp:** Tìm các tổ hợp đặc trưng thường đi cùng với `y = yes`.
- `duration` được loại khỏi phân lớp vì chỉ biết sau khi cuộc gọi kết thúc.

#### 1.3. Từ điển dữ liệu
#### 1.3. Từ điển dữ liệu

Bộ dữ liệu ban đầu gồm 16 thuộc tính đầu vào và 1 thuộc tính mục tiêu `y`,
tổng cộng 17 cột. Sau Feature Engineering, tạo thêm 13 thuộc tính mới,
nâng tổng số cột lên 30.

| Thuộc tính | Ý nghĩa | Kiểu | Thang đo |
|---|---|---|---|
| `age` | Tuổi khách hàng | Số nguyên | Tỷ lệ |
| `job` | Nghề nghiệp | Phân loại | Danh nghĩa |
| `marital` | Tình trạng hôn nhân | Phân loại | Danh nghĩa |
| `education` | Trình độ học vấn | Phân loại | Danh nghĩa |
| `default` | Có nợ tín dụng hay không | Phân loại | Danh nghĩa |
| `balance` | Số dư tài khoản | Số nguyên | Tỷ lệ |
| `housing` | Có vay mua nhà hay không | Phân loại | Danh nghĩa |
| `loan` | Có vay cá nhân hay không | Phân loại | Danh nghĩa |
| `contact` | Hình thức liên lạc | Phân loại | Danh nghĩa |
| `day` | Ngày liên lạc | Số nguyên | Khoảng |
| `month` | Tháng liên lạc | Phân loại | Danh nghĩa |
| `duration` | Thời lượng cuộc gọi | Số nguyên | Tỷ lệ |
| `campaign` | Số lần liên hệ trong chiến dịch | Số nguyên | Tỷ lệ |
| `pdays` | Số ngày từ lần liên hệ trước | Số nguyên | Khoảng |
| `previous` | Số lần liên hệ trước đó | Số nguyên | Tỷ lệ |
| `poutcome` | Kết quả chiến dịch trước | Phân loại | Danh nghĩa |
| `y` | Có đăng ký tiền gửi hay không | Phân loại | Nhị phân |

#### 1.4. Ma trận bộ dữ liệu × kỹ thuật

| Bộ dữ liệu | Phân lớp | Luật kết hợp | Gom cụm |
|---|:---:|:---:|:---:|
| **D3: Bank Marketing** | **X** | **X** |  |



In [ ]:
from pathlib import Path
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import zscore
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

OUT = Path("outputs")
FIGURES = OUT / "figures"
PROCESSED = Path("data/processed")

OUT.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

bank = fetch_ucirepo(id=222)

X_raw = bank.data.features.copy()
y_raw = bank.data.targets.copy()

y_raw = y_raw.iloc[:, 0].rename("y")
df = pd.concat([X_raw, y_raw], axis=1)

df.columns = [str(column).strip().lower() for column in df.columns]

print("Kích thước dữ liệu:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))


### 2. Điều tra giá trị thiếu

Các giá trị `unknown` là một mức của biến phân loại, không tự động xem là missing.
Chỉ các giá trị rỗng hoặc `NaN` mới được thống kê là giá trị thiếu.

In [ ]:
missing = pd.DataFrame({
    "Số lượng thiếu": df.isna().sum(),
    "Tỷ lệ thiếu (%)": (df.isna().mean() * 100).round(2)
}).sort_values(
    "Tỷ lệ thiếu (%)",
    ascending=False
)

display(missing)
missing.to_csv(
    OUT / "missing_report.csv",
    encoding="utf-8-sig"
)


### 3. Nhiễu và ngoại lai

Các biến số có thể chứa giá trị cực trị nhưng chưa chắc là lỗi dữ liệu.
Vì vậy, ngoại lai được phát hiện bằng IQR, Z-score và Boxplot nhưng không tự động xóa.

- `balance`: số dư tài khoản có thể lệch phải.
- `duration`: thời lượng cuộc gọi có thể rất lớn và là leakage nếu dự đoán trước cuộc gọi.
- `campaign`, `previous`: số lần liên hệ có thể chứa một số giá trị cực trị hợp lệ.


In [ ]:
numeric_columns = df.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

outlier_rows = []

for column in numeric_columns:
    values = pd.to_numeric(
        df[column],
        errors="coerce"
    ).dropna()

    if values.empty:
        continue

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    iqr_count = ((values < lower) | (values > upper)).sum()

    if values.std() != 0:
        z_count = (np.abs(zscore(values)) > 3).sum()
    else:
        z_count = 0

    outlier_rows.append({
        "Thuộc tính": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Ngưỡng dưới": lower,
        "Ngưỡng trên": upper,
        "Số ngoại lai IQR": int(iqr_count),
        "Tỷ lệ ngoại lai (%)": round(
            iqr_count / len(values) * 100,
            2
        ),
        "Số ngoại lai Z-score": int(z_count)
    })

outliers = pd.DataFrame(outlier_rows)
display(outliers)

outliers.to_csv(
    OUT / "outlier_report.csv",
    index=False,
    encoding="utf-8-sig"
)

for column in numeric_columns:
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=df[column])
    plt.title(f"Boxplot - {column}")
    plt.tight_layout()
    plt.savefig(
        FIGURES / f"boxplot_{column}.png",
        dpi=120
    )
    plt.show()


### 4. Thêm và biến đổi thuộc tính

Dữ liệu ban đầu có 16 thuộc tính đầu vào và 1 thuộc tính mục tiêu `y`,
tổng cộng 17 cột.

Tạo thêm 13 thuộc tính dẫn xuất, nên dữ liệu sau Feature Engineering có:

- 29 thuộc tính đầu vào.
- 1 thuộc tính mục tiêu `y`.
- Tổng cộng 30 cột.

Các thuộc tính mới gồm:

- `AgeGroup`
- `BalanceGroup`
- `CampaignGroup`
- `PreviousContactGroup`
- `HasAnyLoan`
- `WasContactedBefore`
- `PreviousSuccess`
- `BalanceLog`
- `CampaignLog`
- `PreviousLog`
- `ContactRecencyGroup`
- `ContactIntensity`
- `BalancePerCampaign`

`duration` vẫn được giữ trong dữ liệu khảo sát nhưng bị loại khỏi mô hình
phân lớp vì chỉ biết sau khi cuộc gọi kết thúc, gây rò rỉ thông tin
trong kịch bản dự đoán trước cuộc gọi.

In [ ]:
# Giữ lại 17 thuộc tính gốc và tạo thêm 13 thuộc tính mới

df["AgeGroup"] = pd.cut(
    df["age"],
    bins=[0, 25, 35, 50, 65, np.inf],
    labels=["young", "adult", "middle", "senior", "elderly"]
)

df["BalanceGroup"] = pd.cut(
    df["balance"],
    bins=[-np.inf, 0, 500, 1500, np.inf],
    labels=["negative", "low", "medium", "high"]
)

df["CampaignGroup"] = pd.cut(
    df["campaign"],
    bins=[0, 1, 2, 4, np.inf],
    labels=["first", "low", "medium", "high"]
)

df["PreviousContactGroup"] = pd.cut(
    df["previous"],
    bins=[-1, 0, 1, 3, np.inf],
    labels=["none", "one", "few", "many"]
)

df["HasAnyLoan"] = (
    (df["housing"].astype(str).str.lower() == "yes") |
    (df["loan"].astype(str).str.lower() == "yes")
).astype(int)

df["WasContactedBefore"] = (
    df["pdays"] != -1
).astype(int)

df["PreviousSuccess"] = (
    df["poutcome"].astype(str).str.lower() == "success"
).astype(int)

df["BalanceLog"] = (
    np.sign(df["balance"]) *
    np.log1p(np.abs(df["balance"]))
)

df["CampaignLog"] = np.log1p(df["campaign"])
df["PreviousLog"] = np.log1p(df["previous"])

# Khoảng thời gian từ lần liên hệ trước
df["ContactRecencyGroup"] = pd.cut(
    df["pdays"],
    bins=[-2, 0, 7, 30, 90, np.inf],
    labels=[
        "not_contacted",
        "within_week",
        "within_month",
        "within_quarter",
        "over_quarter"
    ]
)

# Cường độ liên hệ hiện tại so với lịch sử
df["ContactIntensity"] = (
    df["campaign"] / (df["previous"] + 1)
)

# Số dư trung bình trên mỗi lần liên hệ
df["BalancePerCampaign"] = (
    df["balance"] /
    df["campaign"].replace(0, np.nan)
).fillna(0)

engineered_columns = [
    "AgeGroup",
    "BalanceGroup",
    "CampaignGroup",
    "PreviousContactGroup",
    "HasAnyLoan",
    "WasContactedBefore",
    "PreviousSuccess",
    "BalanceLog",
    "CampaignLog",
    "PreviousLog",
    "ContactRecencyGroup",
    "ContactIntensity",
    "BalancePerCampaign"
]

display(df[engineered_columns].head())
display(df[engineered_columns].describe(include="all").T)

print("Số thuộc tính gốc:", 17)
print("Số thuộc tính thêm:", len(engineered_columns))
print("Tổng số thuộc tính:", df.shape[1])

assert len(engineered_columns) == 13
assert df.shape[1] == 30

df.to_csv(
    PROCESSED / "bank_marketing_feature_engineered.csv",
    index=False,
    encoding="utf-8-sig"
)


### 5. Phân lớp nhị phân

Mục tiêu là dự đoán khách hàng có đăng ký tiền gửi hay không.

- `y = yes` được mã hóa thành `1`.
- `y = no` được mã hóa thành `0`.
- `duration` bị loại để tránh rò rỉ dữ liệu.
- Biến phân loại được mã hóa One-Hot.
- Biến số được điền median và chuẩn hóa.
- Sử dụng Logistic Regression và Random Forest.


In [ ]:
X = df.drop(
    columns=["y", "duration"],
    errors="ignore"
)

y = (
    df["y"]
    .astype(str)
    .str.lower()
    .eq("yes")
    .astype(int)
)

numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = [
    column for column in X.columns
    if column not in numeric_features
]

preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

classification_results = []

for model_name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    classification_results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_test,
            probabilities
        )
    })

    matrix = confusion_matrix(
        y_test,
        predictions
    )

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues"
    )
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Dự đoán")
    plt.ylabel("Thực tế")
    plt.tight_layout()
    plt.savefig(
        FIGURES / f"confusion_matrix_{model_name.replace(' ', '_')}.png",
        dpi=140
    )
    plt.show()

classification_results = pd.DataFrame(
    classification_results
).sort_values(
    "F1",
    ascending=False
)

display(classification_results)

classification_results.to_csv(
    OUT / "classification_results.csv",
    index=False,
    encoding="utf-8-sig"
)


### 6. Luật kết hợp

Mỗi khách hàng được biểu diễn thành một tập item:

- `job=...`
- `education=...`
- `AgeGroup=...`
- `BalanceGroup=...`
- `HasAnyLoan=...`
- `PreviousSuccess=...`
- `TARGET=yes/no`

Các luật hướng tới `TARGET=yes` được lọc theo:

- `support >= 0.03`
- `confidence >= 0.50`
- `lift > 1`

Luật kết hợp chỉ mô tả sự đồng xuất hiện, không khẳng định quan hệ nhân quả.


In [ ]:
rule_df = df.copy()

if len(rule_df) > 20000:
    rule_df = rule_df.sample(
        n=20000,
        random_state=42
    ).copy()
selected_columns = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome",
    "AgeGroup",
    "BalanceGroup",
    "CampaignGroup",
    "PreviousContactGroup",
    "ContactRecencyGroup",
    "HasAnyLoan",
    "WasContactedBefore",
    "PreviousSuccess"
]

selected_columns = [
    column for column in selected_columns
    if column in rule_df.columns
]

transactions = []

for _, row in rule_df.iterrows():
    items = []

    for column in selected_columns:
        if pd.notna(row[column]):
            items.append(
                f"{column}={row[column]}"
            )

    target = str(row["y"]).lower()
    items.append(f"TARGET={target}")

    transactions.append(items)

encoder = TransactionEncoder()

encoded = encoder.fit(
    transactions
).transform(
    transactions,
    sparse=True
)

basket = pd.DataFrame.sparse.from_spmatrix(
    encoded,
    columns=encoder.columns_
)

frequent_itemsets = apriori(
    basket,
    min_support=0.03,
    use_colnames=True,
    max_len=3,
    low_memory=True
)

if frequent_itemsets.empty:
    rules = pd.DataFrame()
else:
    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=0.50
    )

if not rules.empty:
    target_yes_rules = rules[
        rules["consequents"].apply(
            lambda items: "TARGET=yes" in items
        )
    ].copy()

    target_yes_rules = target_yes_rules[
        target_yes_rules["lift"] > 1
    ].sort_values(
        ["lift", "confidence"],
        ascending=False
    )

    display(
        target_yes_rules[
            [
                "antecedents",
                "consequents",
                "support",
                "confidence",
                "lift"
            ]
        ].head(30)
    )
else:
    target_yes_rules = pd.DataFrame()
    print("Không tìm thấy luật phù hợp.")

display(
    frequent_itemsets.sort_values(
        "support",
        ascending=False
    ).head(25)
)

frequent_itemsets.to_csv(
    OUT / "frequent_itemsets.csv",
    index=False,
    encoding="utf-8-sig"
)

target_yes_rules.to_csv(
    OUT / "association_rules_target_yes.csv",
    index=False,
    encoding="utf-8-sig"
)


### 7. Lưu dữ liệu và kết luận

#### Kết luận

- Bank Marketing phù hợp với bài toán phân lớp nhị phân.
- Từ 17 thuộc tính gốc, đã tạo thêm 13 thuộc tính mới, tổng cộng 30 thuộc tính.
- Feature Engineering bổ sung thông tin về nhóm tuổi, số dư, khoản vay và lịch sử liên hệ.
- `duration` bị loại khỏi phân lớp vì gây leakage.
- Luật kết hợp giúp tìm các tổ hợp đặc trưng thường đi cùng khách hàng đăng ký tiền gửi.
- Ngoại lai được phát hiện nhưng không tự động xóa vì có thể phản ánh khách hàng thực tế.
- Kết quả được đánh giá bằng Accuracy, Precision, Recall, F1, ROC-AUC,
  Support, Confidence và Lift.


In [ ]:
df.to_csv(
    PROCESSED / "bank_marketing_processed.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu dữ liệu D3.")
print("Kích thước cuối cùng:", df.shape)
print("Số thuộc tính cuối cùng:", df.shape[1])


### 8. Sản phẩm nộp

- `D3/khao-sat-d3-hoan-chinh.ipynb`
- `D3/data/processed/bank_marketing_processed.csv`
- `D3/data/processed/bank_marketing_feature_engineered.csv`
- `D3/outputs/missing_report.csv`
- `D3/outputs/outlier_report.csv`
- `D3/outputs/classification_results.csv`
- `D3/outputs/frequent_itemsets.csv`
- `D3/outputs/association_rules_target_yes.csv`
- `D3/outputs/figures/`
